# Нейронные сети и обработка естественного языка - NLP

# Модуль 1. Элементарные операции и пре-процессинг текстов

1. Загрузка и предварительная обработка текстов
2. Поиск по тексту
3. Полнотекстовый поиск в pandas
4. Полнотекстовый поиск с использованием TF-IDF
5. Полнотекстовый поиск с использованием BM25
6. Токенизация, стемминг и лемматизация

Данный модуль посвящен решению задачи полнотекстового поиска. 

Мы рассмотрим, как влияет на качество поиска предварительная обработка текста с использованием различных алгоритмов.

Задача поиска по тексту является одной из базовых для создания RAG-систем, и алгоритмы, рассмотренные ниже - на сегодняшний день актуальны и востребованы.

Также приемы пре-процесснга текста, рассмотренные в данном модуле, являются базовыми для создания векторных представлений текстов, которые могут быть использованы в прочих задачах NLP.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. Загрузка и предварительная обработка текстов

## 1.1 Загрузка данных с помощью pandas из csv-файла

In [ ]:
df = pd.read_csv("data/demo.csv")
df

In [ ]:
df = pd.read_csv("data/demo_cp1251.csv", 
                 encoding="cp1251", # указать кодировку
                 sep=";", # указать разделитель
                 )
df

### 1.1.1 Неизвестные кодировки

In [ ]:
df = pd.read_csv("data/demo_unknown.csv")
df

In [ ]:
%pip install chardet

In [ ]:
import chardet

with open("data/demo_unknown.csv", "rb") as f:
    raw = f.read(100000)

chardet_result = chardet.detect(raw)

print(chardet_result)

In [ ]:
df = pd.read_csv("data/demo_unknown.csv", encoding=chardet_result["encoding"])
df

## 1.2 Загрузка данных с помощью библиотеки ```datasets```

In [ ]:
%pip install datasets
from datasets import load_dataset

In [ ]:
# можно загружать файлы локально
dataset = load_dataset("csv", data_files="data/demo_unknown.csv", encoding=chardet_result["encoding"])
dataset

In [ ]:
dataset["train"].to_pandas()

In [ ]:
# можно загружать файлы по URL
dataset_url = "https://github.com/easyise/spec_python_courses/raw/refs/heads/master/neural_02_nlp/data/demo_unknown.csv"
dataset_from_url = load_dataset("csv", data_files=dataset_url, encoding=chardet_result["encoding"])
dataset_from_url["train"].to_pandas()

### 1.2.1 Загрузка датасета RuBQ

GitHub: https://github.com/vladislavneon/RuBQ/raw/refs/heads/master/RuBQ_2.0/RuBQ_2.0_paragraphs.json

Размеченный датасет с вопросами, ответами и источниками информации для ответов. Вопросы и ответы на русском языке. Источники информации - тексты из Википедии.


In [ ]:
rubq = load_dataset("json", 
                    data_files="https://github.com/vladislavneon/RuBQ/raw/refs/heads/master/RuBQ_2.0/RuBQ_2.0_paragraphs.json"
                    cache_dir="data/data_cache"
                    )
rubq

In [ ]:
ds = load_dataset(
    "d0rj/RuBQ_2.0-paragraphs",
    cache_dir="data/data_cache",
)

ds

In [ ]:
rubq = ds['paragraphs'].to_pandas().set_index("uid")
rubq

In [ ]:
import seaborn as sns

rubq["paragraph_length"] = rubq["text"].apply(lambda x: len(x.split()))
rubq["paragraph_chars"] = rubq["text"].apply(len)

rubq.describe()

In [ ]:
sns.histplot(rubq[['paragraph_length', 'paragraph_chars']], log_scale=True);

# 2. Поиск по тексту

## 2.1 Простой поиск по тексту

В pandas реализован метод ```str.contains()```, который позволяет искать подстроки в текстовых столбцах. Этот метод поддерживает регулярные выражения, что делает его мощным инструментом для поиска.

In [ ]:
query = "премьер-министр"

found_mask = rubq["text"].str.contains(
        query,
        case=False,   # игнорировать регистр
        na=False      # пропускать NaN
    )
found_mask

In [ ]:
results = rubq[found_mask]

results[["text"]]

In [ ]:
# поиск по регулярному выражению: найти все доменные имена в тексте
domain_pattern = r"\b(?:[a-zA-Z0-9-]+\.)+[a-zA-Z]{2,}\b"
domain_mask = rubq["text"].str.contains(domain_pattern, case=False, na=False)
domain_results = rubq[domain_mask]
domain_results

## 2.2 Поиск с помощью TF-IDF и косинусного сходства

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
texts = rubq["text"].fillna("").astype(str).tolist()

len(texts)

In [ ]:
texts[0][:300]

In [ ]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    max_features=50_000,
    ngram_range=(1, 2)   # слова и пары слов
)

X = vectorizer.fit_transform(texts)

X.shape

X - это матрица TF-IDF, которая представляет тексты в виде числовых векторов. В этой матрице:
- строки = документы / параграфы
- столбцы = слова и биграммы
- значения = TF-IDF веса


In [ ]:
query = "первый президент России"

q_vec = vectorizer.transform([query])

q_vec.shape

In [ ]:
q_vec[:, :1000].toarray()

In [ ]:
# посчитаем количество совпадений между вектором запроса и всеми текстами
scores = cosine_similarity(q_vec, X).ravel()

scores[:10]

In [ ]:
top_k = 5

top_idx = np.argsort(scores)[::-1][:top_k]

top_idx

In [ ]:
for rank, idx in enumerate(top_idx, start=1):
    print("=" * 100)
    print(f"#{rank}")
    print(f"score: {scores[idx]:.4f}")
    print(rubq.iloc[idx]["text"][:1000])

In [ ]:
def search_tfidf(query, top_k=5):
    q_vec = vectorizer.transform([query])
    scores = cosine_similarity(q_vec, X).ravel()
    
    top_idx = np.argsort(scores)[::-1][:top_k]
    
    results = rubq.iloc[top_idx].copy()
    results["score"] = scores[top_idx]
    
    return results[["score", "text"]]

In [ ]:
search_tfidf("первый президент России", top_k=5)

In [ ]:
search_tfidf("столица Франции", top_k=5)

In [ ]:
search_tfidf("Война и Мир", top_k=5)

In [ ]:
search_tfidf("кто написал войну и мир", top_k=5)

In [ ]:
search_tfidf("самая высокая гора", top_k=5)

## 2.3. Поиск с помощью BM25

BM25 - расширение TF-IDF, которое учитывает длину документа и частоту термина в документе. BM25 часто используется в информационном поиске и считается более эффективным, чем TF-IDF для ранжирования документов по релевантности.

Поиск с использованием BM25 - один из компонентов RAG.

In [ ]:
%pip install rank-bm25

In [ ]:
# выполняем примитивную токенизацию: разбиваем текст на слова и приводим к нижнему регистру
def dumb_tokenizer(words):
    return words.lower().split()


tokenized_corpus = [
    dumb_tokenizer(text)
    for text in texts
]

In [ ]:
from rank_bm25 import BM25Okapi

bm25 = BM25Okapi(tokenized_corpus)

In [ ]:
def search_bm25(query, df, bm25=bm25, tokenizer=dumb_tokenizer, top_k=5):

    tokenized_query = tokenizer(query)

    scores = bm25.get_scores(tokenized_query)

    top_idx = np.argsort(scores)[::-1][:top_k]

    results = df.iloc[top_idx].copy()
    results["score"] = scores[top_idx]

    return results[["score", "text"]]

In [ ]:
search_bm25("первый президент России", rubq)

In [ ]:
search_tfidf("первый президент России")

## 2.4 Формулы TF-IDF и BM25

**IDF**

$
IDF(t) = \log \frac{N}{df(t)}
$

где:

- $N$ — количество документов
- $df(t)$ — количество документов, содержащих термин $t$

---

**TF-IDF**

$
TFIDF(t,d) = TF(t,d) \cdot IDF(t)
$

где:

- $TF(t,d)$ — частота термина $t$ в документе $d$
- $IDF(t)$ — обратная частота документа

---

**BM25**

$
BM25(q,d)=\sum_{i=1}^{n}
IDF(q_i)
\cdot
\frac{
f(q_i,d)\cdot(k_1+1)
}{
f(q_i,d)+
k_1\left(
1-b+b\cdot\frac{|d|}{avgdl}
\right)
}
$

где:

- $f(q_i,d)$ — количество вхождений термина $q_i$ в документ
- $|d|$ — длина документа
- $avgdl$ — средняя длина документа
- $k_1$ и $b$ — параметры BM25

# 3. Токенизация, стемминг и лемматизация

Как повлияют токенизация, стемминг и лемматизация на результаты поиска?

Рассмотрим четыре подхода:

1. Токенизация по словам
2. Токенизация с использованием токенизатора для русского языка
3. Токенизация с использованием токенизатора для русского языка + стемминг
4. Токенизация с использованием токенизатора для русского языка + лемматизация

**Что такое стемминг?**
Это приведение слова к его основе (стему) с помощью простых эвристик.
```кошки
кошке
кошкой
→ кошк
```

**Что такое лемматизация?**

Это приведение слова к его начальной форме (лемме).
```
кошки
кошке
кошкой
→ кошка
```




In [ ]:
a_text = rubq.loc[96, 'text']
print("Символов в оригинальном тексте:", len(a_text))
print(a_text)

In [ ]:
# внимательно обратим внимание на содержимое. все ли ок?
dumb_tokenizer(a_text)

## 3.1 Классика: NLTK

NLTK расшифровывается как Natural Language Toolkit. Это одна из самых популярных библиотек для обработки естественного языка в Python. NLTK предоставляет широкий спектр инструментов для токенизации, стемминга, лемматизации и других задач обработки текста.

Для русского языка - не очень. Для английского - отлично.




In [ ]:
%pip install nltk

In [ ]:
import nltk
nltk.download('punkt_tab')

In [ ]:
from nltk.tokenize import word_tokenize

word_tokenize(a_text)

In [ ]:
# напишем токенизатор на основе NLTK
def tokenizer_nltk(text):
    # уберем все не-слова
    tok = [word.lower() for word in word_tokenize(text) if word.isalnum() or len(word) > 1]
    return tok

tokenizer_nltk(a_text)

## 3.2 Токенизатор для русского языка: библиотека razdel

In [ ]:
# используем токенизатор для русского языка razdel
%pip install razdel


In [ ]:
import razdel
for x in razdel.tokenize(a_text):
    print(x)
    print(f"токен: '{x.text}', позиция: {x.start}-{x.stop}")
    break


In [ ]:
[token.text.lower() for token in razdel.tokenize(a_text) if token.text.isalnum() or len(token.text) > 1]

In [ ]:
def tokenizer_razdel(text):
    return [token.text.lower() for token in razdel.tokenize(text) if token.text.isalnum() or len(token.text) > 1]

set(tokenizer_razdel(a_text)) == set(tokenizer_nltk(a_text))

In [ ]:
set(tokenizer_razdel(a_text)) - set(tokenizer_nltk(a_text))

In [ ]:
diff_mask = rubq.iloc[:100,:]['text'].apply(lambda x: set(tokenizer_razdel(x)) != set(tokenizer_nltk(x)))
diff_mask.value_counts()

In [ ]:
rubq[:100][diff_mask]

**ПРАКТИКА**

Поэкспериментируйте с различными текстами и попробуйте понять, в чем разница между токенизаторами.

На чем они ошибаются?

Как должен работать идеальный токенизатор?


In [ ]:
# ваш код здесь





## 3.3 Стемминг

Используем токенизатор razdel и стеммер Snowball для русского языка

In [ ]:
from nltk.stem.snowball import RussianStemmer, EnglishStemmer

stemmer_rus = RussianStemmer()
stemmer_eng = EnglishStemmer()

str_eng = "A quick brown fox jumps over the lazy dog."
str_rus = "Быстрая коричневая лиса перепрыгивает через ленивую собаку."

print("Стемминг английского текста:")
for word in tokenizer_nltk(str_eng):
    print(f"{word} -> {stemmer_eng.stem(word)}")
print("\nСтемминг русского текста:")
for word in tokenizer_razdel(str_rus):
    print(f"{word} -> {stemmer_rus.stem(word)}")

In [ ]:
def tokenizer_stemmer(text, language="russian"):
    if language == "russian":
        stemmer = RussianStemmer()
        tokens = tokenizer_razdel(text)
    elif language == "english":
        stemmer = EnglishStemmer()
        tokens = tokenizer_nltk(text)
    else:
        raise ValueError("Unsupported language")
    
    return [stemmer.stem(token) for token in tokens]

a_text = rubq.loc[42, 'text']
tokenizer_stemmer(a_text)

## 3.4 Лемматизация

Будем использовать библиотеку pymorphy3 для лемматизации русского языка.

In [ ]:
%pip install pymorphy3

In [ ]:
import pymorphy3

morph = pymorphy3.MorphAnalyzer()

print(str_rus)
for word in tokenizer_razdel(str_rus):
    print(f"{word} -> {morph.parse(word)[0].normal_form}")



In [ ]:
print(str_eng)
for word in tokenizer_nltk(str_eng):
    print(f"{word} -> {morph.parse(word)[0].normal_form}")

**Для английского языка**: WordNetLemmatizer из библиотеки NLTK.

In [ ]:
nltk.download("wordnet")
nltk.download("omw-1.4")

In [ ]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

words = [
    "cars",
    "running",
    "better",
    "children"
]

for word in words:
    print(f"{word} -> {lemmatizer.lemmatize(word)}")

In [ ]:

def tokenizer_lemmas(text, language="russian"):
    if language == "russian":
        morph = pymorphy3.MorphAnalyzer()
        lemmatizer = lambda token: morph.parse(token)[0].normal_form
        tokens = tokenizer_razdel(text)
    elif  language == "english":
        lemmatizer = WordNetLemmatizer().lemmatize
        tokens = tokenizer_nltk(text)
    else:
        raise ValueError("Unsupported language")
    
    return [lemmatizer(token) for token in tokens]


tokenizer_lemmas(a_text, language="russian")

In [ ]:
# сравним качество поиска в BM25
search_bm25("первый президент России", rubq) # dumb tokenizer

In [ ]:
bm25_nltk = BM25Okapi([tokenizer_nltk(text) for text in texts])
search_bm25("первый президент России", rubq, bm25=bm25_nltk, tokenizer=tokenizer_nltk) # nltk tokenizer

In [ ]:
bm25_razdel = BM25Okapi([tokenizer_razdel(text) for text in texts])
search_bm25("первый президент России", rubq, bm25=bm25_razdel, tokenizer=tokenizer_razdel) # razdel tokenizer

In [ ]:

bm25_stemmer = BM25Okapi([tokenizer_stemmer(text) for text in texts])
search_bm25("первый президент России", rubq, bm25=bm25_stemmer, tokenizer=tokenizer_stemmer) # stemmer tokenizer

In [ ]:
bm25_lemmas = BM25Okapi([tokenizer_lemmas(text) for text in texts])
search_bm25("первый президент России", rubq, bm25=bm25_lemmas, tokenizer=tokenizer_lemmas) # lemmatizer tokenizer

## 3.5 Использование токенизатора на базе Transformers

In [ ]:
%pip install -q transformers

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "DeepPavlov/rubert-base-cased"
)

In [ ]:
text = "Первый президент России управлял государством"

tokens = tokenizer.tokenize(text)

print(tokens)

In [ ]:
text = "электрифицировавшийся"

print(tokenizer.tokenize(text))

In [ ]:
[tokenize_transformer(text) for text in texts[:100]]

In [ ]:
def tokenize_transformer(text):

    return tokenizer.tokenize(
        text.lower()
    )

bm25_transformer = BM25Okapi([tokenize_transformer(text) for text in texts])

In [ ]:
search_bm25("первый президент России", rubq, bm25=bm25_transformer, tokenizer=tokenize_transformer) # transformer tokenizer

**ПРАКТИКА**

1. Загрузите любой из следующих датасетов:
- [Lenta.ru (short)](https://huggingface.co/datasets/zloelias/lenta-ru-short)
- [IMDB](https://huggingface.co/datasets/imdb)
- [Habr (subset)](https://huggingface.co/datasets/bikingSolo/IlyaGusev_habr_subset)
- [Russian Toxic Comments](https://huggingface.co/datasets/AlexSham/Toxic_Russian_Comments)
- любой другой датасет с текстами на русском/английском языках

2. Реализуйте полнотекстовый поиск по текстам из этих датасетов с помощью BM25, используя различные токенизаторы (обычный, razdel, стемминг, лемматизация). Сравните результаты и сделайте выводы о том, какой токенизатор работает лучше для вашего датасета.

In [ ]:
# ваш код здесь





